In [ ]:
!nvidia-smi

In [ ]:
%cd /kaggle/working

!git clone https://github.com/emanroshan/DPO.git

%cd /kaggle/working/DPO

In [ ]:
!python -m pip uninstall -y -q torchao || true

!python -m pip install -q "transformers>=4.43.0,<5" "datasets>=2.18,<5" huggingface_hub accelerate tqdm pyyaml sentencepiece safetensors pytest

!python -m pip install -q -e . --no-deps

In [ ]:
import torch
import transformers
import datasets

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)

print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

for i in range(torch.cuda.device_count()):
    print(f"GPU {i}:", torch.cuda.get_device_name(i))

In [ ]:
from pathlib import Path
import shutil

ROOT = Path("/kaggle/working/DPO")
ROOT.mkdir(parents=True, exist_ok=True)

# Find SFT model in attached saved notebook output
matches = list(
    Path("/kaggle/input").rglob(
        "DPO/outputs/sft/best/model.safetensors"
    )
)

print("Found checkpoints:", matches)

assert matches, (
    "SFT checkpoint not found! "
    "Attach notebook7313e97c61 under Add Input."
)

saved_root = matches[0].parents[3]

for folder in ["outputs/sft", "outputs/baseline", "eval_sets"]:
    src = saved_root / folder
    dst = ROOT / folder

    if src.exists():
        shutil.copytree(src, dst, dirs_exist_ok=True)
        print("Restored:", folder)

SFT_BEST = ROOT / "outputs/sft/best"

assert (SFT_BEST / "model.safetensors").exists()
assert (SFT_BEST / "tokenizer_config.json").exists()

print("SFT restored successfully!")

In [ ]:
import os
import json
import math
import random
import gc
import time
import shutil

from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F

from datasets import load_dataset
from torch.utils.data import Dataset, DataLoader, Subset
from tqdm.auto import tqdm

ROOT = Path("/kaggle/working/DPO")
OUTPUTS = ROOT / "outputs"
OUTPUTS.mkdir(parents=True, exist_ok=True)

SEED = 42

print("Phase 4 setup ready!")

In [ ]:
from transformers import AutoTokenizer

SFT_BEST = "/kaggle/working/DPO/outputs/sft/best"

tokenizer = AutoTokenizer.from_pretrained(
    SFT_BEST,
    fix_mistral_regex=True
)

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"

print("Tokenizer ready:", type(tokenizer).__name__)

In [ ]:
from pathlib import Path
import shutil

# Locate the saved notebook output
matches = list(
    Path("/kaggle/input").rglob(
        "DPO/outputs/sft/best/model.safetensors"
    )
)

print("Found SFT checkpoints:", matches)

assert len(matches) == 1, (
    "SFT checkpoint not found. "
    "Make sure notebook7313e97c61 is attached as an input."
)

# Root of the saved DPO repository
saved_repo = matches[0].parents[3]

# Current working repository
working_repo = Path("/kaggle/working/DPO")
working_repo.mkdir(parents=True, exist_ok=True)

# Restore SFT outputs and baseline outputs
for folder in ["sft", "baseline"]:
    source = saved_repo / "outputs" / folder
    destination = working_repo / "outputs" / folder

    if source.exists():
        shutil.copytree(
            source,
            destination,
            dirs_exist_ok=True
        )
        print("Restored:", folder)

# Restore HH evaluation prompts if present
source_eval = saved_repo / "eval_sets"
if source_eval.exists():
    shutil.copytree(
        source_eval,
        working_repo / "eval_sets",
        dirs_exist_ok=True
    )

SFT_BEST = working_repo / "outputs/sft/best"

assert (SFT_BEST / "model.safetensors").exists()

print("\nSFT checkpoint ready:", SFT_BEST)
print("Model size:",
      round((SFT_BEST / "model.safetensors").stat().st_size / 1024**2),
      "MB")

In [ ]:

from pathlib import Path
import shutil

ROOT = Path("/kaggle/working/DPO")
ROOT.mkdir(parents=True, exist_ok=True)

matches = list(Path("/kaggle/input").rglob(
    "DPO/outputs/sft/best/model.safetensors"
))

print("Saved SFT models found:", matches)
assert len(matches) == 1, (
    "Attach notebook7313e97c61 as a Kaggle notebook input."
)

SAVED_ROOT = matches[0].parents[3]
print("Saved repository:", SAVED_ROOT)

for relative in [
    "outputs/sft",
    "outputs/baseline",
    "eval_sets",
]:
    src = SAVED_ROOT / relative
    dst = ROOT / relative
    if src.exists():
        shutil.copytree(src, dst, dirs_exist_ok=True)
        print("Restored:", relative)

SFT_BEST = ROOT / "outputs/sft/best"

assert (SFT_BEST / "model.safetensors").is_file()
assert (SFT_BEST / "config.json").is_file()

print("BEST SFT MODEL READY:", SFT_BEST)
print("Baseline restored:",
      (ROOT / "outputs/baseline/base_generations.json").exists())


In [ ]:

%pip -q install "transformers>=4.45,<5" "datasets>=3,<5" "accelerate>=1,<2" bitsandbytes safetensors matplotlib

import os, gc, json, math, random, time
import numpy as np
import torch
import torch.nn.functional as F
import bitsandbytes as bnb
import matplotlib.pyplot as plt

from pathlib import Path
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from torch.utils.data import Dataset, DataLoader, Subset
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

assert torch.cuda.device_count() >= 2, (
    "Select Kaggle Accelerator: GPU T4 x2"
)

for i in range(torch.cuda.device_count()):
    print(i, torch.cuda.get_device_name(i))

OUTPUTS = ROOT / "outputs"
OUTPUTS.mkdir(exist_ok=True)

tokenizer = AutoTokenizer.from_pretrained(str(SFT_BEST))
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

BETA = 0.1
LR = 1e-6
MAX_PROMPT = 256
MAX_LENGTH = 512
GRAD_ACCUM = 32
EPOCHS = 1

print("Phase 4 setup complete.")


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import bitsandbytes as bnb

BETA = 0.1
LR = 1e-6
GRAD_ACCUM = 32
MAX_PROMPT = 256
MAX_LENGTH = 512
EPOCHS = 1

print("DPO hyperparameters ready!")

In [ ]:

def save_jsonl(rows, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

def read_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

def valid_pair(prompt, chosen, rejected):
    return (
        isinstance(prompt, str)
        and isinstance(chosen, str)
        and isinstance(rejected, str)
        and bool(prompt.strip())
        and bool(chosen.strip())
        and bool(rejected.strip())
        and chosen.strip() != rejected.strip()
    )

def clean_pair(prompt, chosen, rejected):
    return {
        "prompt": prompt.strip(),
        "chosen": chosen.strip(),
        "rejected": rejected.strip()
    }

def extract_hh(text):
    """Strictly use a single Human -> Assistant turn."""
    if not isinstance(text, str):
        return None
    if text.count("\n\nHuman:") != 1:
        return None
    if text.count("\n\nAssistant:") != 1:
        return None
    if not text.startswith("\n\nHuman:"):
        return None

    rest = text[len("\n\nHuman:"):]
    if "\n\nAssistant:" not in rest:
        return None

    prompt, response = rest.split("\n\nAssistant:", 1)
    return prompt.strip(), response.strip()

def hh_to_pair(row):
    c = extract_hh(row["chosen"])
    r = extract_hh(row["rejected"])
    if c is None or r is None:
        return None
    if c[0] != r[0]:
        return None
    if not valid_pair(c[0], c[1], r[1]):
        return None
    return clean_pair(c[0], c[1], r[1])

def message_text(message):
    if isinstance(message, str):
        return message
    if isinstance(message, dict):
        content = message.get("content", "")
        return content if isinstance(content, str) else ""
    return ""

def uf_to_pair(row):
    chosen = row.get("chosen")
    rejected = row.get("rejected")

    if not (
        isinstance(chosen, list)
        and isinstance(rejected, list)
        and len(chosen) == 2
        and len(rejected) == 2
    ):
        return None

    if chosen[0].get("role") != "user":
        return None
    if rejected[0].get("role") != "user":
        return None
    if chosen[1].get("role") != "assistant":
        return None
    if rejected[1].get("role") != "assistant":
        return None

    prompt = message_text(chosen[0])
    if prompt != message_text(rejected[0]):
        return None

    good = message_text(chosen[1])
    bad = message_text(rejected[1])

    if not valid_pair(prompt, good, bad):
        return None
    return clean_pair(prompt, good, bad)

def collect_pairs(dataset, converter, count, seed=42):
    shuffled = dataset.shuffle(seed=seed)
    rows = []
    for example in shuffled:
        pair = converter(example)
        if pair is not None:
            rows.append(pair)
            if len(rows) == count:
                break

    assert len(rows) == count, (
        f"Only found {len(rows)} valid pairs; need {count}."
    )
    return rows


In [ ]:

HH_DIR = OUTPUTS / "dpo_hh"
HH_TRAIN = HH_DIR / "train_15000.jsonl"
HH_VAL = HH_DIR / "val_1000.jsonl"

if HH_TRAIN.exists() and HH_VAL.exists():
    print("Using existing HH-RLHF pair files.")
else:
    hh_train_raw = load_dataset(
        "Anthropic/hh-rlhf", split="train"
    )
    hh_val_raw = load_dataset(
        "Anthropic/hh-rlhf", split="test"
    )

    hh_train = collect_pairs(
        hh_train_raw, hh_to_pair, 15000, SEED
    )
    hh_val = collect_pairs(
        hh_val_raw, hh_to_pair, 1000, SEED
    )

    save_jsonl(hh_train, HH_TRAIN)
    save_jsonl(hh_val, HH_VAL)

print("HH train:", len(read_jsonl(HH_TRAIN)))
print("HH validation:", len(read_jsonl(HH_VAL)))


In [ ]:

UF_DIR = OUTPUTS / "dpo_ultrafeedback"
UF_TRAIN = UF_DIR / "train_15000.jsonl"
UF_VAL = UF_DIR / "val_1000.jsonl"

if UF_TRAIN.exists() and UF_VAL.exists():
    print("Using existing UltraFeedback pair files.")
else:
    uf_train_raw = load_dataset(
        "HuggingFaceH4/ultrafeedback_binarized",
        split="train_prefs"
    )
    uf_val_raw = load_dataset(
        "HuggingFaceH4/ultrafeedback_binarized",
        split="test_prefs"
    )

    uf_train = collect_pairs(
        uf_train_raw, uf_to_pair, 15000, SEED
    )
    uf_val = collect_pairs(
        uf_val_raw, uf_to_pair, 1000, SEED
    )

    save_jsonl(uf_train, UF_TRAIN)
    save_jsonl(uf_val, UF_VAL)

print("UltraFeedback train:", len(read_jsonl(UF_TRAIN)))
print("UltraFeedback validation:", len(read_jsonl(UF_VAL)))


In [ ]:

def tokenize_pair(row):
    prompt = (
        "### Instruction:\n"
        + row["prompt"]
        + "\n\n### Response:\n"
    )

    prompt_ids = tokenizer.encode(
        prompt, add_special_tokens=False
    )[:MAX_PROMPT]

    def encode_response(text):
        response_ids = tokenizer.encode(
            text, add_special_tokens=False
        )

        response_ids = response_ids[
            :MAX_LENGTH - len(prompt_ids) - 1
        ]
        response_ids = response_ids + [tokenizer.eos_token_id]

        ids = prompt_ids + response_ids
        labels = [-100] * len(prompt_ids) + response_ids

        return {
            "input_ids": ids,
            "attention_mask": [1] * len(ids),
            "labels": labels
        }

    return {
        "chosen": encode_response(row["chosen"]),
        "rejected": encode_response(row["rejected"])
    }

class PreferenceDataset(Dataset):
    def __init__(self, path):
        self.rows = read_jsonl(path)

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        return tokenize_pair(self.rows[idx])

def dpo_collate(items):
    result = {}

    for side in ("chosen", "rejected"):
        max_len = max(len(x[side]["input_ids"]) for x in items)

        for field in ("input_ids", "attention_mask", "labels"):
            pad = (
                tokenizer.pad_token_id if field == "input_ids"
                else 0 if field == "attention_mask"
                else -100
            )

            result[f"{side}_{field}"] = torch.tensor(
                [
                    x[side][field]
                    + [pad] * (max_len - len(x[side][field]))
                    for x in items
                ],
                dtype=torch.long
            )

    return result

hh_train_loader = DataLoader(
    PreferenceDataset(HH_TRAIN),
    batch_size=1, shuffle=True,
    collate_fn=dpo_collate
)

hh_val_loader = DataLoader(
    PreferenceDataset(HH_VAL),
    batch_size=1, shuffle=False,
    collate_fn=dpo_collate
)

uf_train_loader = DataLoader(
    PreferenceDataset(UF_TRAIN),
    batch_size=1, shuffle=True,
    collate_fn=dpo_collate
)

uf_val_loader = DataLoader(
    PreferenceDataset(UF_VAL),
    batch_size=1, shuffle=False,
    collate_fn=dpo_collate
)

assert len(hh_train_loader.dataset) == 15000
assert len(uf_train_loader.dataset) == 15000
assert len(hh_val_loader.dataset) == 1000
assert len(uf_val_loader.dataset) == 1000

print("All four DPO dataloaders ready.")


In [ ]:

def sequence_logprobs(model, ids, mask, labels):
    outputs = model(
        input_ids=ids,
        attention_mask=mask,
        use_cache=False
    )

    logits = outputs.logits[:, :-1, :].float()
    targets = labels[:, 1:]

    valid = targets != -100
    safe_targets = targets.masked_fill(~valid, 0)

    selected = logits.gather(
        -1, safe_targets.unsqueeze(-1)
    ).squeeze(-1)

    log_partition = torch.logsumexp(logits, dim=-1)

    token_logps = selected - log_partition
    return token_logps.masked_fill(~valid, 0).sum(dim=-1)

def get_pair_logps(model, batch, device):
    def score(side):
        return sequence_logprobs(
            model,
            batch[f"{side}_input_ids"].to(device),
            batch[f"{side}_attention_mask"].to(device),
            batch[f"{side}_labels"].to(device)
        )

    chosen = score("chosen")
    rejected = score("rejected")
    return chosen, rejected

def dpo_loss_from_logps(pc, pr, rc, rr, beta=0.1):
    chosen_reward = beta * (pc - rc)
    rejected_reward = beta * (pr - rr)

    margin = chosen_reward - rejected_reward
    loss = -F.logsigmoid(margin).mean()

    accuracy = (margin > 0).float().mean()
    reward_margin = margin.mean()

    return loss, accuracy, reward_margin

# Mathematical sanity check
z = torch.zeros(1)
loss, acc, margin = dpo_loss_from_logps(z, z, z, z)

assert abs(loss.item() - math.log(2)) < 1e-5
assert acc.item() == 0.0
assert margin.item() == 0.0

print("DPO objective passed sanity check.")


In [ ]:

def load_dpo_models():
    print("Loading trainable SFT policy on cuda:0...")

    policy = AutoModelForCausalLM.from_pretrained(
        str(SFT_BEST),
        torch_dtype=torch.float32,
        low_cpu_mem_usage=True
    ).to("cuda:0")

    policy.config.use_cache = False
    policy.gradient_checkpointing_enable()
    policy.train()

    print("Loading frozen SFT reference on cuda:1...")

    reference = AutoModelForCausalLM.from_pretrained(
        str(SFT_BEST),
        torch_dtype=torch.float16,
        low_cpu_mem_usage=True
    ).to("cuda:1")

    reference.config.use_cache = False
    reference.requires_grad_(False)
    reference.eval()

    return policy, reference


In [ ]:

@torch.no_grad()
def evaluate_dpo(policy, reference, val_loader, beta=0.1):
    policy.eval()
    reference.eval()

    total_loss = 0.0
    total_correct = 0
    total_margin = 0.0
    total_pairs = 0

    for batch in tqdm(val_loader, desc="DPO validation", leave=False):
        with torch.autocast("cuda", dtype=torch.float16):
            pc, pr = get_pair_logps(policy, batch, "cuda:0")

        with torch.autocast("cuda", dtype=torch.float16):
            rc, rr = get_pair_logps(reference, batch, "cuda:1")

        pc, pr = pc.float(), pr.float()
        rc = rc.float().to("cuda:0")
        rr = rr.float().to("cuda:0")

        loss, accuracy, margin = dpo_loss_from_logps(
            pc, pr, rc, rr, beta
        )

        n = pc.numel()
        total_pairs += n
        total_loss += loss.item() * n
        total_correct += accuracy.item() * n
        total_margin += margin.item() * n

    policy.train()

    return {
        "val_loss": total_loss / total_pairs,
        "reward_accuracy": total_correct / total_pairs,
        "reward_margin": total_margin / total_pairs
    }


In [ ]:

def save_dpo_checkpoint(model, folder):
    folder = Path(folder)
    folder.mkdir(parents=True, exist_ok=True)

    model.save_pretrained(
        str(folder),
        safe_serialization=True,
        max_shard_size="2GB"
    )
    tokenizer.save_pretrained(str(folder))

    assert any(folder.glob("*.safetensors"))
    print("Checkpoint saved:", folder)

def write_json(path, data):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w") as f:
        json.dump(data, f, indent=2)

def write_jsonl(path, rows):
    with open(path, "w") as f:
        for row in rows:
            f.write(json.dumps(row) + "\n")


In [ ]:

def train_dpo(
    run_name,
    train_loader,
    val_loader,
    beta=0.1,
    lr=1e-6,
    grad_accum=32,
    epochs=1,
    eval_every=150
):
    out = OUTPUTS / run_name
    out.mkdir(parents=True, exist_ok=True)

    summary_path = out / "training_summary.json"

    # Prevent accidental reruns of completed experiments.
    if summary_path.exists():
        summary = json.loads(summary_path.read_text())
        if summary.get("completed"):
            print("Already completed:", run_name)
            return summary

    policy = None
    reference = None
    optimizer = None

    try:
        policy, reference = load_dpo_models()

        optimizer = bnb.optim.AdamW8bit(
            policy.parameters(),
            lr=lr,
            weight_decay=0.0,
            min_8bit_size=4096
        )

        scaler = torch.amp.GradScaler(
            "cuda",
            init_scale=128.0,
            growth_interval=2000
        )

        expected_updates = math.ceil(
            len(train_loader) / grad_accum
        ) * epochs

        print("Experiment:", run_name)
        print("Training pairs:", len(train_loader.dataset))
        print("Expected updates:", expected_updates)

        history = []
        successful_steps = 0
        skipped_steps = 0
        best_accuracy = -1.0
        best_step = None

        optimizer.zero_grad(set_to_none=True)

        progress = tqdm(
            total=expected_updates,
            desc=f"{run_name} optimizer updates"
        )

        def validate_and_record(step):
            nonlocal best_accuracy, best_step

            metrics = evaluate_dpo(
                policy, reference, val_loader, beta
            )

            record = {"step": step, **metrics}
            history.append(record)

            print(
                f"Step {step} | "
                f"Val loss: {metrics['val_loss']:.4f} | "
                f"Reward accuracy: {metrics['reward_accuracy']:.4f} | "
                f"Reward margin: {metrics['reward_margin']:.4f}"
            )

            if metrics["reward_accuracy"] > best_accuracy:
                best_accuracy = metrics["reward_accuracy"]
                best_step = step

                save_dpo_checkpoint(policy, out / "best")
                print("Saved new best DPO checkpoint.")

            write_jsonl(out / "metrics.jsonl", history)

        for epoch in range(epochs):
            accumulated = 0

            for batch_idx, batch in enumerate(train_loader):
                with torch.autocast(
                    device_type="cuda",
                    dtype=torch.float16
                ):
                    pc, pr = get_pair_logps(
                        policy, batch, "cuda:0"
                    )

                with torch.no_grad():
                    with torch.autocast(
                        device_type="cuda",
                        dtype=torch.float16
                    ):
                        rc, rr = get_pair_logps(
                            reference, batch, "cuda:1"
                        )

                loss, accuracy, margin = dpo_loss_from_logps(
                    pc.float(),
                    pr.float(),
                    rc.float().to("cuda:0"),
                    rr.float().to("cuda:0"),
                    beta
                )

                if not torch.isfinite(loss).item():
                    raise RuntimeError(
                        "Nonfinite DPO loss. Training stopped."
                    )

                scaler.scale(loss).backward()
                accumulated += 1

                last_batch = (batch_idx + 1 == len(train_loader))

                if accumulated == grad_accum or last_batch:
                    scaler.unscale_(optimizer)

                    # Average gradients across actual microbatches.
                    for p in policy.parameters():
                        if p.grad is not None:
                            p.grad.div_(accumulated)

                    grad_norm = torch.nn.utils.clip_grad_norm_(
                        policy.parameters(),
                        max_norm=1.0,
                        error_if_nonfinite=False
                    )

                    old_scale = scaler.get_scale()
                    scaler.step(optimizer)
                    scaler.update()
                    new_scale = scaler.get_scale()

                    optimizer.zero_grad(set_to_none=True)
                    accumulated = 0

                    if new_scale < old_scale:
                        skipped_steps += 1
                        print(
                            f"Skipped update #{skipped_steps} "
                            f"(grad_norm={float(grad_norm):.4g}, "
                            f"scale={old_scale:g}->{new_scale:g})"
                        )

                        if successful_steps == 0 and skipped_steps >= 5:
                            raise RuntimeError(
                                "Five consecutive optimizer updates "
                                "were skipped. Diagnose FP16 gradients "
                                "before full training."
                            )
                        continue

                    successful_steps += 1
                    progress.update(1)

                    if successful_steps % eval_every == 0:
                        validate_and_record(successful_steps)

        progress.close()

        if successful_steps == 0:
            raise RuntimeError(
                "ZERO optimizer updates succeeded. "
                "Do not start a full DPO experiment."
            )

        # Always validate and save at the end.
        if not history or history[-1]["step"] != successful_steps:
            validate_and_record(successful_steps)

        save_dpo_checkpoint(policy, out / "final")

        summary = {
            "run_name": run_name,
            "completed": True,
            "training_pairs": len(train_loader.dataset),
            "validation_pairs": len(val_loader.dataset),
            "epochs": epochs,
            "beta": beta,
            "learning_rate": lr,
            "gradient_accumulation": grad_accum,
            "successful_optimizer_updates": successful_steps,
            "skipped_optimizer_updates": skipped_steps,
            "expected_optimizer_updates": expected_updates,
            "best_step": best_step,
            "best_reward_accuracy": best_accuracy,
            "initial_checkpoint": str(SFT_BEST),
            "best_checkpoint": str(out / "best"),
            "final_checkpoint": str(out / "final"),
            "optimizer": "bitsandbytes AdamW8bit"
        }

        write_json(summary_path, summary)

        print("\nTRAINING COMPLETED:", run_name)
        print(json.dumps(summary, indent=2))

        return summary

    finally:
        del optimizer, reference, policy
        gc.collect()
        torch.cuda.empty_cache()


In [ ]:

test_loader = DataLoader(
    Subset(hh_train_loader.dataset, range(32)),
    batch_size=1,
    shuffle=False,
    collate_fn=dpo_collate
)

test_summary = train_dpo(
    run_name="dpo_hh_memory_test",
    train_loader=test_loader,
    val_loader=hh_val_loader,
    beta=BETA,
    lr=LR,
    grad_accum=GRAD_ACCUM,
    epochs=1,
    eval_every=150
)

assert test_summary["successful_optimizer_updates"] == 1, (
    "32-pair test failed to complete an optimizer update."
)

print("32-pair test PASSED. Ready for full DPO.")


In [ ]:

hh_summary = train_dpo(
    run_name="dpo_hh",
    train_loader=hh_train_loader,
    val_loader=hh_val_loader,
    beta=0.1,
    lr=1e-6,
    grad_accum=32,
    epochs=1,
    eval_every=150
)

print("HH-RLHF completed.")
print("Best reward accuracy:",
      hh_summary["best_reward_accuracy"])
print("Successful updates:",
      hh_summary["successful_optimizer_updates"])


In [ ]:

uf_summary = train_dpo(
    run_name="dpo_ultrafeedback",
    train_loader=uf_train_loader,
    val_loader=uf_val_loader,
    beta=0.1,
    lr=1e-6,
    grad_accum=32,
    epochs=1,
    eval_every=150
)

print("UltraFeedback completed.")
print("Best reward accuracy:",
      uf_summary["best_reward_accuracy"])
print("Successful updates:",
      uf_summary["successful_optimizer_updates"])


In [ ]:

def plot_dpo_metrics(run_name):
    folder = OUTPUTS / run_name
    rows = read_jsonl(folder / "metrics.jsonl")

    steps = [r["step"] for r in rows]

    metrics = [
        ("val_loss", "Validation DPO loss"),
        ("reward_accuracy", "Validation reward accuracy"),
        ("reward_margin", "Validation reward margin")
    ]

    for key, title in metrics:
        plt.figure(figsize=(7, 4))
        plt.plot(
            steps,
            [r[key] for r in rows],
            marker="o"
        )
        plt.xlabel("Optimizer updates")
        plt.ylabel(title)
        plt.title(f"{run_name}: {title}")
        plt.grid(alpha=0.3)
        plt.tight_layout()
        plt.savefig(
            folder / f"{key}.png",
            dpi=160
        )
        plt.show()
        plt.close()

plot_dpo_metrics("dpo_hh")
plot_dpo_metrics("dpo_ultrafeedback")


In [ ]:

for run_name in ["dpo_hh", "dpo_ultrafeedback"]:
    folder = OUTPUTS / run_name

    required = [
        folder / "best/model.safetensors",
        folder / "final/model.safetensors",
        folder / "metrics.jsonl",
        folder / "training_summary.json",
        folder / "val_loss.png",
        folder / "reward_accuracy.png",
        folder / "reward_margin.png"
    ]

    for path in required:
        assert path.exists(), f"Missing: {path}"

    summary = json.loads(
        (folder / "training_summary.json").read_text()
    )

    print("\n", run_name)
    print("Training pairs:", summary["training_pairs"])
    print("Validation pairs:", summary["validation_pairs"])
    print("Successful updates:",
          summary["successful_optimizer_updates"])
    print("Skipped updates:",
          summary["skipped_optimizer_updates"])
    print("Best reward accuracy:",
          summary["best_reward_accuracy"])

print("\nALL PHASE 4 OUTPUT FILES VERIFIED.")


In [ ]:
%pip -q install "transformers>=4.45,<5" safetensors accelerate

import json
import random
import shutil
import gc
import torch

from pathlib import Path
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

ROOT = Path("/kaggle/working/DPO")
EVAL_DIR = ROOT / "outputs/eval"
EVAL_DIR.mkdir(parents=True, exist_ok=True)

print("GPUs:", torch.cuda.device_count())

In [ ]:
from pathlib import Path
import shutil

ROOT = Path("/kaggle/working/DPO")
SOURCE = Path(
    "/kaggle/input/notebooks/emanroshan/notebook3b4c81ea04/DPO"
)

assert SOURCE.exists(), "Attach notebook3b4c81ea04 first!"

folders = [
    "outputs/sft",
    "outputs/dpo_hh",
    "outputs/dpo_ultrafeedback",
    "outputs/baseline",
    "eval_sets"
]

for folder in folders:
    src = SOURCE / folder
    dst = ROOT / folder

    assert src.exists(), f"Missing folder: {src}"

    shutil.copytree(src, dst, dirs_exist_ok=True)
    print("Restored:", folder)

SFT_BEST = ROOT / "outputs/sft/best"
HH_BEST = ROOT / "outputs/dpo_hh/best"
UF_BEST = ROOT / "outputs/dpo_ultrafeedback/best"

for name, path in [
    ("SFT", SFT_BEST),
    ("HH-DPO", HH_BEST),
    ("UltraFeedback-DPO", UF_BEST)
]:
    assert (path / "model.safetensors").exists()
    print(name, "READY")

print("\nAll checkpoints and evaluation datasets are ready!")

In [ ]:
import json
import gc
import random
import torch

from pathlib import Path
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM

ROOT = Path("/kaggle/working/DPO")
EVAL_DIR = ROOT / "outputs/eval"
EVAL_DIR.mkdir(parents=True, exist_ok=True)

SFT_BEST = ROOT / "outputs/sft/best"
HH_BEST = ROOT / "outputs/dpo_hh/best"
UF_BEST = ROOT / "outputs/dpo_ultrafeedback/best"

HH_FILE = ROOT / "eval_sets/hh_rlhf_eval_300_seed42.json"
ALPACA_FILE = ROOT / "outputs/baseline/alpacaeval_300_seed42.json"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

MAX_INPUT_TOKENS = 512
MAX_NEW_TOKENS = 256

print("Device:", DEVICE)
print("Evaluation folder:", EVAL_DIR)

In [ ]:
def read_records(path):
    path = Path(path)

    if path.suffix == ".jsonl":
        with open(path, encoding="utf-8") as f:
            return [
                json.loads(line)
                for line in f
                if line.strip()
            ]

    with open(path, encoding="utf-8") as f:
        data = json.load(f)

    if isinstance(data, list):
        return data

    if isinstance(data, dict):
        for key in ["data", "examples", "prompts", "items"]:
            if isinstance(data.get(key), list):
                return data[key]

    raise ValueError(f"Unexpected dataset format: {path}")


hh_rows = read_records(HH_FILE)
alpaca_rows = read_records(ALPACA_FILE)

print("HH evaluation examples:", len(hh_rows))
print("AlpacaEval examples:", len(alpaca_rows))

print("\nHH example:")
print(json.dumps(hh_rows[0], indent=2, ensure_ascii=False)[:1800])

print("\nAlpacaEval example:")
print(json.dumps(alpaca_rows[0], indent=2, ensure_ascii=False)[:1800])

In [ ]:
def extract_prompt(row):
    if isinstance(row, str):
        return row.strip()

    if not isinstance(row, dict):
        raise ValueError(f"Unsupported row: {type(row)}")

    for key in [
        "prompt",
        "instruction",
        "input_text",
        "question",
        "query"
    ]:
        value = row.get(key)

        if isinstance(value, str) and value.strip():
            return value.strip()

    raise ValueError(
        f"Cannot extract prompt. Available keys: {list(row.keys())}"
    )


hh_prompts = [extract_prompt(row) for row in hh_rows]
alpaca_prompts = [extract_prompt(row) for row in alpaca_rows]

assert len(hh_prompts) == 300
assert len(alpaca_prompts) == 300

print("HH prompts:", len(hh_prompts))
print("Alpaca prompts:", len(alpaca_prompts))

print("\nFirst HH prompt:")
print(hh_prompts[0][:1000])

print("\nFirst Alpaca prompt:")
print(alpaca_prompts[0][:1000])

In [ ]:
sft_train = ROOT / "outputs/sft/train_30000.jsonl"
hh_train = ROOT / "outputs/dpo_hh/train_15000.jsonl"
uf_train = ROOT / "outputs/dpo_ultrafeedback/train_15000.jsonl"

for name, path in [
    ("SFT", sft_train),
    ("HH-DPO", hh_train),
    ("UltraFeedback-DPO", uf_train)
]:
    print(f"\n===== {name} TRAINING EXAMPLE =====")

    with open(path, encoding="utf-8") as f:
        row = json.loads(f.readline())

    print(json.dumps(row, indent=2, ensure_ascii=False)[:2000])

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(
    str(SFT_BEST),
    local_files_only=True
)

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token


def format_prompt(prompt, dataset):
    prompt = prompt.strip()

    if dataset == "hh":
        # Preserve the existing HH conversation structure.
        if prompt.endswith("Assistant:"):
            return prompt

        if prompt.startswith("Human:"):
            return prompt + "\n\nAssistant:"

    # Plain instruction prompts use the saved chat template.
    if tokenizer.chat_template:
        return tokenizer.apply_chat_template(
            [{"role": "user", "content": prompt}],
            tokenize=False,
            add_generation_prompt=True
        )

    return f"Human: {prompt}\nAssistant:"


def load_model(checkpoint):
    print("Loading:", checkpoint)

    model = AutoModelForCausalLM.from_pretrained(
        str(checkpoint),
        torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32,
        local_files_only=True
    )

    model = model.to(DEVICE)
    model.eval()

    return model


@torch.inference_mode()
def generate_response(model, prompt, dataset):
    formatted = format_prompt(prompt, dataset)

    inputs = tokenizer(
        formatted,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_INPUT_TOKENS
    ).to(DEVICE)

    output = model.generate(
        **inputs,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=False,
        repetition_penalty=1.1,
        no_repeat_ngram_size=3,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id
    )

    new_tokens = output[0, inputs["input_ids"].shape[1]:]

    response = tokenizer.decode(
        new_tokens,
        skip_special_tokens=True
    ).strip()

    return response


def unload_model(model):
    del model
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


print("Tokenizer and generation functions ready.")

In [ ]:
model = load_model(SFT_BEST)

for i in range(3):
    print(f"\n{'=' * 60}")
    print("HH PROMPT", i + 1)
    print(hh_prompts[i][:500])

    response = generate_response(
        model,
        hh_prompts[i],
        "hh"
    )

    print("\nSFT RESPONSE:")
    print(response[:1200])

unload_model(model)

In [ ]:
def save_jsonl(path, rows):
    with open(path, "w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")


def generate_dataset(
    checkpoint,
    prompts,
    dataset,
    model_name,
    output_filename
):
    output_path = EVAL_DIR / output_filename

    completed = {}

    if output_path.exists():
        with open(output_path, encoding="utf-8") as f:
            for line in f:
                if line.strip():
                    row = json.loads(line)
                    completed[row["id"]] = row

    print(
        f"\n{model_name}: "
        f"{len(completed)}/{len(prompts)} already completed"
    )

    if len(completed) == len(prompts):
        print("Already finished:", output_path)
        return output_path

    model = load_model(checkpoint)

    try:
        with open(output_path, "a", encoding="utf-8") as f:
            for i, prompt in enumerate(tqdm(prompts, desc=model_name)):
                if i in completed:
                    continue

                response = generate_response(
                    model,
                    prompt,
                    dataset
                )

                record = {
                    "id": i,
                    "dataset": dataset,
                    "model": model_name,
                    "prompt": prompt,
                    "response": response
                }

                f.write(
                    json.dumps(record, ensure_ascii=False) + "\n"
                )
                f.flush()

    finally:
        unload_model(model)

    print("Saved:", output_path)
    return output_path

In [ ]:
sft_hh_file = generate_dataset(
    checkpoint=SFT_BEST,
    prompts=hh_prompts,
    dataset="hh",
    model_name="sft",
    output_filename="sft_hh.jsonl"
)

dpo_hh_file = generate_dataset(
    checkpoint=HH_BEST,
    prompts=hh_prompts,
    dataset="hh",
    model_name="dpo_hh",
    output_filename="dpo_hh.jsonl"
)

print("HH generation completed!")

In [ ]:
sft_alpaca_file = generate_dataset(
    checkpoint=SFT_BEST,
    prompts=alpaca_prompts,
    dataset="alpacaeval",
    model_name="sft",
    output_filename="sft_alpacaeval.jsonl"
)

dpo_uf_file = generate_dataset(
    checkpoint=UF_BEST,
    prompts=alpaca_prompts,
    dataset="alpacaeval",
    model_name="dpo_ultrafeedback",
    output_filename="dpo_ultrafeedback_alpacaeval.jsonl"
)

print("UltraFeedback generation completed!")

In [ ]:
def load_by_id(path):
    rows = read_records(path)
    result = {}

    for row in rows:
        result[int(row["id"])] = row

    return result


def create_pairs(
    sft_file,
    dpo_file,
    dataset,
    pair_filename,
    mapping_filename,
    seed=42
):
    sft = load_by_id(sft_file)
    dpo = load_by_id(dpo_file)

    assert set(sft) == set(dpo), "Mismatched example IDs"

    rng = random.Random(seed)

    pairs = []
    mapping = []

    for i in sorted(sft):
        assert sft[i]["prompt"] == dpo[i]["prompt"]

        if rng.random() < 0.5:
            response_a = sft[i]["response"]
            response_b = dpo[i]["response"]
            model_a = "sft"
            model_b = dpo[i]["model"]
        else:
            response_a = dpo[i]["response"]
            response_b = sft[i]["response"]
            model_a = dpo[i]["model"]
            model_b = "sft"

        pair_id = f"{dataset}_{i:04d}"

        pairs.append({
            "pair_id": pair_id,
            "prompt": sft[i]["prompt"],
            "response_a": response_a,
            "response_b": response_b
        })

        mapping.append({
            "pair_id": pair_id,
            "model_a": model_a,
            "model_b": model_b
        })

    pair_path = EVAL_DIR / pair_filename
    mapping_path = EVAL_DIR / mapping_filename

    save_jsonl(pair_path, pairs)
    save_jsonl(mapping_path, mapping)

    print("Created", len(pairs), "pairs")
    print("Judge file:", pair_path)
    print("Private mapping:", mapping_path)

    return pair_path, mapping_path


hh_pairs, hh_mapping = create_pairs(
    sft_hh_file,
    dpo_hh_file,
    dataset="hh",
    pair_filename="hh_pairs.jsonl",
    mapping_filename="hh_hidden_mapping.jsonl"
)

uf_pairs, uf_mapping = create_pairs(
    sft_alpaca_file,
    dpo_uf_file,
    dataset="alpacaeval",
    pair_filename="ultrafeedback_pairs.jsonl",
    mapping_filename="ultrafeedback_hidden_mapping.jsonl"
)

In [ ]:
expected_files = [
    "sft_hh.jsonl",
    "dpo_hh.jsonl",
    "sft_alpacaeval.jsonl",
    "dpo_ultrafeedback_alpacaeval.jsonl",
    "hh_pairs.jsonl",
    "hh_hidden_mapping.jsonl",
    "ultrafeedback_pairs.jsonl",
    "ultrafeedback_hidden_mapping.jsonl"
]

print("PHASE 5 OUTPUT VERIFICATION\n")

for filename in expected_files:
    path = EVAL_DIR / filename

    if path.exists():
        rows = read_records(path)
        status = "PASS" if len(rows) == 300 else "INCOMPLETE"
        print(f"{status}: {filename} — {len(rows)} rows")
    else:
        print(f"MISSING: {filename}")

print("\nOutput folder:", EVAL_DIR)

In [ ]:
from pathlib import Path
import json

EVAL_DIR = Path("/kaggle/working/DPO/outputs/eval")

HH_PAIRS = EVAL_DIR / "hh_pairs.jsonl"
UF_PAIRS = EVAL_DIR / "ultrafeedback_pairs.jsonl"

assert HH_PAIRS.exists(), "HH pairs missing"
assert UF_PAIRS.exists(), "UltraFeedback pairs missing"

def read_jsonl(path):
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

hh_pairs = read_jsonl(HH_PAIRS)
uf_pairs = read_jsonl(UF_PAIRS)

print("HH pairs:", len(hh_pairs))
print("UltraFeedback pairs:", len(uf_pairs))

assert len(hh_pairs) == 300
assert len(uf_pairs) == 300

print("\nTask 7 evaluation files ready!")